In [ ]:
from robovast.common.analysis import (CampaignDataError, config_file,
                                      get_behavior_info, read_runs, read_table)
# Public, but not re-exported from the package, so it comes from the submodule. Imported
# rather than reimplemented: which clock a pose may be differentiated against is the
# library's rule, and a second copy of it here is a copy that goes stale.
from robovast.common.analysis.metrics import pose_time_base
import pandas as pd
# `robovast_nav.map_visualizer`, not `robovast_nav.gui`: the gui package went with the Qt
# desktop tools, and the matplotlib visualizer moved up beside its siblings.
from robovast_nav.map_visualizer import MapVisualizer
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import os
DATA_DIR = ''

# These notebooks serve every .vast in this directory, and those split into two families:
# five scenarios drive `robot.follow_waypoints`, and scenario_nav_through_poses.osc drives
# scenario_nav_through_poses.osc drive `robot.nav_through_poses`. The behaviour is
# recorded under `<actor type>.<action>`, so a name hardcoded to either family leaves the
# other with empty duration panels and no error -- read it from the tree instead.
NAV_ACTIONS = ('nav_through_poses', 'follow_waypoints', 'nav_to_pose')


def navigation_behavior(behaviors):
    """The navigation behaviour's name as this campaign recorded it."""
    names = set(behaviors['behavior_name'].dropna())
    for action in NAV_ACTIONS:
        found = sorted(n for n in names if n.rsplit('.', 1)[-1] == action)
        if found:
            return found[0]
    raise SystemExit(
        f"No navigation behaviour here. Looked for {', '.join(NAV_ACTIONS)}; the tree "
        f"recorded: {', '.join(sorted(n for n in names if '.' in n)) or '(none)'}.")


# One read per table, so the message names the one that is missing. Together in a single
# try, a campaign with poses but no behaviour tree -- execution.bt_log off, or an execution
# image whose scenario_execution predates --bt-log -- reported "no postprocessed data",
# which is a different problem with a different fix.
try:
    df = read_table(DATA_DIR, "poses", with_params=True,
                    require=['frame', 'timestamp', 'position.x', 'position.y',
                             'orientation.yaw'])
except CampaignDataError as e:
    print(e)
    raise SystemExit("This campaign has no postprocessed data to analyse.")

try:
    df_behaviors = read_table(DATA_DIR, "behaviors",
                              require=['behavior_name', 'status_name', 'timestamp'])
except CampaignDataError as e:
    # Every panel from here on is keyed off the navigation behaviour, so there is no
    # partial view worth rendering -- but the reason is not the one above.
    print(e)
    raise SystemExit(
        "This campaign recorded no behaviour tree, so its navigation cannot be timed or "
        "named. See the message above for the two ways that happens.")

if df.empty:
    raise SystemExit("This campaign recorded no poses in any run.")

# Generate colors for each run
runs = df['run_id'].unique()
configs = df['config_name'].unique()
# One colour per run, used by every per-run panel below.
colors = cm.rainbow(np.linspace(0, 1, len(runs)))
run_to_color = dict(zip(runs, colors))
# One colour per configuration, used by the per-config comparison plots below. It was
# referenced there but never assigned, so those cells raised NameError on any campaign.
config_colors = cm.rainbow(np.linspace(0, 1, len(configs)))

# `timestamp` is when the message ARRIVED and stays the join key; `stamp` is when the pose
# was true, and is the only one an interval may be computed from.
TIME = pose_time_base(df)
if TIME != 'stamp':
    print("No publisher timestamps in this campaign: intervals below are arrival times, "
          "quantized by /clock and jittered by delivery.")

NAV_BEHAVIOR = navigation_behavior(df_behaviors)
print(f"Navigation behaviour: {NAV_BEHAVIOR}.")
df_behavior_info = get_behavior_info(NAV_BEHAVIOR, df_behaviors)

# Per-run outcome from the campaign's own runs table, keyed like every other frame
# here. read_run_statuses reads test.xml instead, which is the right source when there
# is no data.db -- but this notebook cannot render without one anyway.
df_statuses = read_runs(DATA_DIR)

# Runs that recorded no poses are absent from every pose-derived panel below, while the
# summary table counts every run the campaign executed. Reconciled out loud, so the two
# numbers do not silently disagree.
_analysed = set(map(tuple, df[['config_name', 'run_id']].drop_duplicates().values))
_executed = set(map(tuple, df_statuses[['config_name', 'run_id']].values))
_no_poses = _executed - _analysed
if _no_poses:
    print(f"{len(_no_poses)} of {len(_executed)} run(s) recorded no poses and are absent "
          f"from the pose-derived panels, though the summary counts them.")

## Overview

In [ ]:
from IPython.display import display, Markdown

n_configs = df_statuses['config_name'].nunique()
n_runs_total = len(df_statuses)
n_success = (df_statuses['status'] == 'passed').sum()
n_failure = (df_statuses['status'] == 'failed').sum()
n_unknown = (df_statuses['status'] == 'unknown').sum()
failure_pct = 100 * n_failure / n_runs_total if n_runs_total > 0 else 0

# Distance traveled from ground truth poses
df_gt = df[df['frame'] == 'nav2_turtlebot4_base_link_gt'].copy()
distance_per_run = {}
for (config, run), group in df_gt.groupby(['config_name', 'run_id'], observed=False):
    if len(group) < 2:
        continue
    # Ordered by the measurement clock: a path summed in arrival order
    # double-counts wherever two samples landed in one tick out of order.
    g = group.sort_values(TIME)
    dx = np.diff(g['position.x'].values)
    dy = np.diff(g['position.y'].values)
    distance_per_run[(config, run)] = float(np.sqrt(dx**2 + dy**2).sum())

total_distance = sum(distance_per_run.values())

unknown_row = f"| Unknown | **{n_unknown}** |\n" if n_unknown > 0 else ""
display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Configs | **{n_configs}** |
| Runs total | **{n_runs_total}** |
| Passed | **{n_success}** |
| Failed | **{n_failure} ({failure_pct:.1f}%)** |
{unknown_row}| Total distance driven (GT) | **{total_distance:.1f} m** |
"""))

## Per-config breakdown
#rows = []
#for config, grp in df_statuses.groupby('config_name'):
#    passed = (grp['status'] == 'passed').sum()
#    failed = (grp['status'] == 'failed').sum()
#    total = len(grp)
#    rows.append(f"| {config} | {passed} | {failed} | {total} |")

#display(Markdown(
#    "### Per-config pass/fail\n\n"
#    "| Config | Passed | Failed | Total |\n"
#    "|--------|--------|--------|-------|\n" +
#    "\n".join(rows)
#))

In [ ]:
# Reduced-to-same-maps view: hash map image, group paths by map
import hashlib
import yaml

def _map_image_path_from_yaml(yaml_path):
    """Resolve map image path from map.yaml. Returns path to .pgm/.png or None."""
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    image_name = data.get('image')
    if not image_name:
        return None
    yaml_dir = os.path.dirname(os.path.abspath(yaml_path))
    return os.path.join(yaml_dir, image_name)

def map_content_hash(map_yaml_path):
    """Quick hash of map content (image file bytes) to detect identical maps."""
    img_path = _map_image_path_from_yaml(map_yaml_path)
    if not img_path or not os.path.isfile(img_path):
        return None
    h = hashlib.md5()
    with open(img_path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

# (config, run) -> (map_path, map_hash); map_file is relative to the campaign's _config/
config_run_map = {}
for config in configs:
    for run in runs:
        df_ct = df[(df['config_name'] == config) & (df['run_id'] == run)]
        if len(df_ct) == 0:
            continue
        map_file0 = df_ct['map_file'].iloc[0]
        map_path = config_file(DATA_DIR, map_file0, config, must_exist=False)
        if not os.path.isfile(map_path):
            continue
        mhash = map_content_hash(map_path)
        if mhash is not None:
            config_run_map[(config, run)] = (map_path, mhash)

# Group by map hash: map_hash -> [(config, run), ...]
from collections import defaultdict
hash_to_runs = defaultdict(list)
for (config, run), (map_path, mhash) in config_run_map.items():
    hash_to_runs[mhash].append((config, run))

# One subplot per unique map; draw all paths that use that map
num_maps = len(hash_to_runs)
cols = min(3, num_maps)
rows = (num_maps + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 7*rows))
if num_maps == 1:
    axes = np.array([axes])
axes = axes.flatten()

path_colors = cm.rainbow(np.linspace(0, 1, max(1, len(configs) * len(runs))))
color_idx = 0
# `map_runs`, not `runs`: this used to shadow the campaign-wide `runs` list, leaving it
# bound to (config, run) tuples for every cell below, which then compared a tuple
# against the run column. Latent until the map path resolved -- an empty
# hash_to_runs never entered the loop.
for i, (mhash, map_runs) in enumerate(hash_to_runs.items()):
    config0, run0 = map_runs[0]
    map_path = config_run_map[(config0, run0)][0]
    viz = MapVisualizer()
    try:
        viz.load_map(map_path)
        viz.create_figure(ax=axes[i], figsize=(7, 7))
    except Exception as e:
        axes[i].set_title(f'Same map ({len(map_runs)} runs)\n(map unavailable)', fontsize=10, fontweight='bold')
        axes[i].axis('off')
        color_idx += len(map_runs)
        continue
    labels = []
    for (config, run) in map_runs:
        df_run = df[(df['config_name'] == config) & (df['run_id'] == run)]
        df_gt_mask = df_run['frame'] == 'nav2_turtlebot4_base_link_gt'
        path_robot = list(zip(df_run.loc[df_gt_mask, 'position.x'],
                             df_run.loc[df_gt_mask, 'position.y']))
        if len(path_robot) > 0:
            c = path_colors[color_idx % len(path_colors)]
            viz.draw_path(path_robot, color=c, linewidth=1.5, alpha=0.6, show_endpoints=False)
            labels.append(f'{config} / {run}')
            color_idx += 1
    viz.ax.set_title(f'Same map ({len(map_runs)} runs)\\n' + ', '.join(labels[:3]) + ('...' if len(labels) > 3 else ''),
                     fontsize=10, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


## Navigation Duration Analysis

In [ ]:
# Multi-config + Multi-Run Navigation Duration Analysis
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Use df_behavior_info directly - it already has one row per run-config combination
df_duration_summary = df_behavior_info[['config_name', 'run_id', 'duration']].copy()

# One row per (config, run) is the shape the panels below assume. A scenario that
# navigates twice in one run breaks that -- pivot would raise rather than plot -- so it is
# reported and reduced to the total the run spent navigating.
_per_run = df_duration_summary.groupby(['config_name', 'run_id'], observed=True).size()
if (_per_run > 1).any():
    print(f"{int((_per_run > 1).sum())} run(s) completed {NAV_BEHAVIOR} more than once; "
          f"the panels below use each run's total navigation time.")
    df_duration_summary = (df_duration_summary
                           .groupby(['config_name', 'run_id'], as_index=False, observed=True)
                           ['duration'].sum())

# Create a pivot table for easier access
duration_pivot = df_duration_summary.pivot(index='config_name', columns='run_id', values='duration')

# Sort configs to match our color scheme
duration_pivot = duration_pivot.reindex(configs)

# 1. Grouped Bar Chart - Duration by config and Run
ax = axes[0, 0]
x = np.arange(len(runs))
width = 0.12  # Width of bars
offsets = np.linspace(-3*width, 3*width, len(configs))

sorted_runs_list = sorted(runs)
for i, config in enumerate(configs):
    config_data = df_duration_summary[df_duration_summary['config_name'] == config].copy()
    if len(config_data) > 0:
        # Create array with NaN for missing runs
        durations = []
        for run in sorted_runs_list:
            run_data = config_data[config_data['run_id'] == run]
            if len(run_data) > 0:
                durations.append(run_data['duration'].values[0])
            else:
                durations.append(np.nan)
        
        # Only plot non-NaN values
        durations = np.array(durations)
        valid_mask = ~np.isnan(durations)
        if valid_mask.any():
            ax.bar(x[valid_mask] + offsets[i], durations[valid_mask], width, label=config, 
                   color=config_colors[i], alpha=0.8, edgecolor='black', linewidth=0.8)

ax.set_xlabel('Run Number', fontsize=11, fontweight='bold')
ax.set_ylabel('Duration (seconds)', fontsize=11, fontweight='bold')
ax.set_title('Navigation Duration: configs × Runs', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(r) for r in sorted_runs_list])
ax.legend(fontsize=8, ncol=2, loc='upper left')
ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)

# 2. Box Plot - Duration Distribution by config
ax = axes[0, 1]
config_duration_list = [df_duration_summary[df_duration_summary['config_name'] == v]['duration'].values 
                         for v in configs]
bp = ax.boxplot(config_duration_list, tick_labels=configs, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], config_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)

for element in ['whiskers', 'fliers', 'medians', 'caps']:
    plt.setp(bp[element], color='black', linewidth=1.5)

ax.set_xlabel('config', fontsize=11, fontweight='bold')
ax.set_ylabel('Duration (seconds)', fontsize=11, fontweight='bold')
ax.set_title('Duration Distribution by config', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Heatmap - Duration across configs and Runs
ax = axes[0, 2]
duration_matrix = duration_pivot.values
im = ax.imshow(duration_matrix, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(np.arange(len(sorted_runs_list)))
ax.set_yticks(np.arange(len(configs)))
ax.set_xticklabels([str(r) for r in sorted_runs_list])
ax.set_yticklabels(configs)
ax.set_xlabel('Run Number', fontsize=11, fontweight='bold')
ax.set_ylabel('config', fontsize=11, fontweight='bold')
ax.set_title('Duration Heatmap: configs × Runs', fontsize=12, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Duration (s)', fontsize=10, fontweight='bold')

# Add text annotations
if duration_matrix.size > 0 and len(sorted_runs_list) > 0:
    for i in range(len(configs)):
        for j in range(len(sorted_runs_list)):
            if not np.isnan(duration_matrix[i, j]):
                text = ax.text(j, i, f'{duration_matrix[i, j]:.1f}',
                              ha="center", va="center", color="black", fontsize=7, fontweight='bold')

# 4. Mean Duration by config with Error Bars
ax = axes[1, 0]
config_means = [df_duration_summary[df_duration_summary['config_name'] == v]['duration'].mean() 
                 for v in configs]
config_stds = [df_duration_summary[df_duration_summary['config_name'] == v]['duration'].std() 
                for v in configs]
x_pos = np.arange(len(configs))

bars = ax.bar(x_pos, config_means, yerr=config_stds, capsize=5, 
              alpha=0.75, color=config_colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('config', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Duration (seconds)', fontsize=11, fontweight='bold')
ax.set_title('Mean Duration by config (±1 SD)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels
for i, (mean, std) in enumerate(zip(config_means, config_stds)):
    ax.text(i, mean + std + 0.5, f'{mean:.1f}s', ha='center', va='bottom', 
            fontsize=9, fontweight='bold')

# 5. Coefficient of Variation by config
ax = axes[1, 1]
config_cv = [(df_duration_summary[df_duration_summary['config_name'] == v]['duration'].std() / 
               df_duration_summary[df_duration_summary['config_name'] == v]['duration'].mean()) * 100 
              for v in configs]

bars = ax.bar(x_pos, config_cv, alpha=0.75, color=config_colors, 
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('config', fontsize=11, fontweight='bold')
ax.set_ylabel('Coefficient of Variation (%)', fontsize=11, fontweight='bold')
ax.set_title('Duration Variability by config', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels
for i, cv in enumerate(config_cv):
    ax.text(i, cv + 0.2, f'{cv:.1f}%', ha='center', va='bottom', 
            fontsize=9, fontweight='bold')

# 6. Violin Plot - Duration Distribution by config
ax = axes[1, 2]

# Filter out empty arrays and track valid positions
valid_data = []
valid_positions = []
valid_colors = []
for i, (data, config) in enumerate(zip(config_duration_list, configs)):
    if len(data) > 0:
        valid_data.append(data)
        valid_positions.append(i)
        valid_colors.append(config_colors[i])

if len(valid_data) > 0:
    parts = ax.violinplot(valid_data, positions=valid_positions, 
                          showmeans=True, showmedians=True)

    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(valid_colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)

    plt.setp(parts['cmeans'], color='red', linewidth=2)
    plt.setp(parts['cmedians'], color='blue', linewidth=2)
    plt.setp(parts['cbars'], color='black', linewidth=1.5)
    plt.setp(parts['cmaxes'], color='black', linewidth=1.5)
    plt.setp(parts['cmins'], color='black', linewidth=1.5)

ax.set_xlabel('config', fontsize=11, fontweight='bold')
ax.set_ylabel('Duration (seconds)', fontsize=11, fontweight='bold')
ax.set_title('Duration Distribution Shape by config', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(configs)))
ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*80)
print("MULTI-config DURATION ANALYSIS SUMMARY")
print("="*80)
for config in configs:
    durations = df_duration_summary[df_duration_summary['config_name'] == config]['duration'].values
    if len(durations) > 0:
        print(f"\n{config}:")
        print(f"  Mean: {durations.mean():.2f}s (±{durations.std():.2f}s)")
        print(f"  Range: [{durations.min():.2f}s, {durations.max():.2f}s]")
        print(f"  CV: {(durations.std()/durations.mean())*100:.1f}%")
    else:
        print(f"\n{config}:")
        print(f"  No data available")
print("="*80)

In [ ]:
# Prepare data for multi-config + multi-run comparison
import numpy as np
import matplotlib.pyplot as plt

# Split by frame and prepare for all runs
df_robot = df[df['frame'] == 'base_link'].copy()
df_groundtruth = df[df['frame'] == 'nav2_turtlebot4_base_link_gt'].copy()

# Sort by config, run and timestamp
# Sorted on TIME: ordering by arrival time leaves rows within one tick in
# arbitrary order, which is enough to hand interp1d a non-monotonic x.
df_robot.sort_values(['config_name', 'run_id', TIME], inplace=True)
df_groundtruth.sort_values(['config_name', 'run_id', TIME], inplace=True)

# print(f"Processing {len(configs)} configs × {len(runs)} runs = {len(configs)*len(runs)} runs")
# print(f"Robot data points: {len(df_robot)}")
# print(f"Ground truth data points: {len(df_groundtruth)}")

In [ ]:
# Calculate errors for each config-run combination
from scipy.interpolate import interp1d

# Store results for each config-run run
run_errors = {}
run_stats = []

for config in configs:
    for run in runs:
        # Get data for this specific config-run combination
        mask_robot = (df_robot['config_name'] == config) & (df_robot['run_id'] == run)
        mask_gt = (df_groundtruth['config_name'] == config) & (df_groundtruth['run_id'] == run)
        
        df_robot_run = df_robot[mask_robot].copy()
        df_gt_run = df_groundtruth[mask_gt].copy()
        
        if len(df_robot_run) == 0 or len(df_gt_run) == 0:
            continue
        
        # Find common time range
        time_start = max(df_robot_run[TIME].iloc[0], df_gt_run[TIME].iloc[0])
        time_end = min(df_robot_run[TIME].iloc[-1], df_gt_run[TIME].iloc[-1])
        
        # Create interpolation functions for ground truth
        gt_timestamps = df_gt_run[TIME].values
        gt_interp_x = interp1d(gt_timestamps, df_gt_run['position.x'].values, kind='linear', fill_value='extrapolate')
        gt_interp_y = interp1d(gt_timestamps, df_gt_run['position.y'].values, kind='linear', fill_value='extrapolate')
        gt_interp_yaw = interp1d(gt_timestamps, df_gt_run['orientation.yaw'].values, kind='linear', fill_value='extrapolate')
        
        # Filter robot data to common time range
        time_mask = (df_robot_run[TIME] >= time_start) & (df_robot_run[TIME] <= time_end)
        df_robot_aligned = df_robot_run[time_mask].copy()
        
        if len(df_robot_aligned) == 0:
            continue
        
        robot_timestamps = df_robot_aligned[TIME].values
        
        # Interpolate ground truth at robot timestamps
        gt_x = gt_interp_x(robot_timestamps)
        gt_y = gt_interp_y(robot_timestamps)
        gt_yaw = gt_interp_yaw(robot_timestamps)
        
        # Calculate position errors
        position_error_x = df_robot_aligned['position.x'].values - gt_x
        position_error_y = df_robot_aligned['position.y'].values - gt_y
        absolute_position_error = np.sqrt(position_error_x**2 + position_error_y**2)
        
        # Calculate orientation error
        orientation_error = df_robot_aligned['orientation.yaw'].values - gt_yaw
        orientation_error = np.arctan2(np.sin(orientation_error), np.cos(orientation_error))
        
        # Store errors for this run
        run_key = f"{config}_{run}"
        run_errors[run_key] = {
            'config_name': config,
            'run_id': run,
            'timestamps': robot_timestamps,
            'position_error': absolute_position_error,
            'position_error_x': position_error_x,
            'position_error_y': position_error_y,
            'orientation_error': orientation_error
        }
        
        # Calculate statistics
        run_stats.append({
            'config_name': config,
            'run_id': run,
            'mean_pos_error': np.mean(absolute_position_error),
            'std_pos_error': np.std(absolute_position_error),
            'max_pos_error': np.max(absolute_position_error),
            'median_pos_error': np.median(absolute_position_error),
            'p95_pos_error': np.percentile(absolute_position_error, 95),
            'mean_orient_error': np.mean(np.abs(orientation_error)),
            'std_orient_error': np.std(orientation_error),
            'max_orient_error': np.max(np.abs(orientation_error))
        })

# Convert to DataFrame for easy analysis
stats_df = pd.DataFrame(run_stats)
# print(f"Calculated errors for {len(run_stats)} runs")


In [ ]:
from robovast.common.analysis import calculate_speeds_from_poses
df_gt_speeds = calculate_speeds_from_poses(df_groundtruth)

In [ ]:
# Speed Statistics by config and Run
speed_stats = []
for config in configs:
    for run in runs:
        mask = (df_gt_speeds['config_name'] == config) & (df_gt_speeds['run_id'] == run)
        df_run_speeds = df_gt_speeds[mask]
        
        if len(df_run_speeds) == 0:
            continue
        
        linear_speeds = df_run_speeds['linear_speed'].values
        angular_speeds = df_run_speeds['angular_speed'].values
        
        speed_stats.append({
            'config_name': config,
            'run_id': run,
            'mean_linear': np.mean(linear_speeds),
            'max_linear': np.max(linear_speeds),
            'std_linear': np.std(linear_speeds),
            'median_linear': np.median(linear_speeds),
            'mean_angular': np.mean(np.abs(angular_speeds)),
            'max_angular': np.max(np.abs(angular_speeds)),
            'std_angular': np.std(angular_speeds),
            'median_angular': np.median(np.abs(angular_speeds))
        })

speed_stats_df = pd.DataFrame(speed_stats)
# print(f"Calculated speed stats for {len(speed_stats)} runs")

## Localization Errors

In [ ]:
# Multi-config Position Error Analysis
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Box Plot - Position Error by config
ax = axes[0, 0]
if 'config_name' in stats_df.columns:
    config_pos_errors = [stats_df[stats_df['config_name'] == v]['mean_pos_error'].values for v in configs]
    bp = ax.boxplot(config_pos_errors, tick_labels=configs, patch_artist=True, widths=0.6)

    for patch, color in zip(bp['boxes'], config_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.5)

    for element in ['whiskers', 'fliers', 'medians', 'caps']:
        plt.setp(bp[element], color='black', linewidth=1.5)

    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Position Error (m)', fontsize=11, fontweight='bold')
    ax.set_title('Position Error Distribution by config', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 2. Heatmap - Mean Position Error by config and Run
ax = axes[0, 1]
if 'config_name' in stats_df.columns and 'run_id' in stats_df.columns:
    # NaN, not zero: a (config, run) cell with no measurement is left blank by imshow,
    # where 0.0 would read as a perfect run. And the run ids are used as ids -- stringifying
    # them made every mask below compare str to int, so nothing ever matched and the whole
    # heatmap rendered as zeros, which is a wrong answer rather than a missing one.
    error_matrix = np.full((len(configs), len(runs)), np.nan)
    sorted_runs_list = sorted(runs)
    for i, config in enumerate(configs):
        for j, run in enumerate(sorted_runs_list):
            mask = (stats_df['config_name'] == config) & (stats_df['run_id'] == run)
            if mask.any():
                error_matrix[i, j] = stats_df[mask]['mean_pos_error'].values[0]

    im = ax.imshow(error_matrix, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(np.arange(len(sorted_runs_list)))
    ax.set_yticks(np.arange(len(configs)))
    ax.set_xticklabels([str(r) for r in sorted_runs_list])
    ax.set_yticklabels(configs)
    ax.set_xlabel('Run Number', fontsize=11, fontweight='bold')
    ax.set_ylabel('config', fontsize=11, fontweight='bold')
    ax.set_title('Mean Position Error Heatmap', fontsize=12, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean Error (m)', fontsize=10, fontweight='bold')

    # Add text annotations
    if error_matrix.size > 0 and len(sorted_runs_list) > 0:
        for i in range(len(configs)):
            for j in range(len(sorted_runs_list)):
                if error_matrix[i, j] > 0:
                    ax.text(j, i, f'{error_matrix[i, j]:.3f}',
                        ha="center", va="center", color="black", fontsize=7, fontweight='bold')

# 3. Mean Error by config with Error Bars
ax = axes[0, 2]
if 'config_name' in stats_df.columns:
    config_mean_errors = [stats_df[stats_df['config_name'] == v]['mean_pos_error'].mean() for v in configs]
    config_std_errors = [stats_df[stats_df['config_name'] == v]['mean_pos_error'].std() for v in configs]
    x_pos = np.arange(len(configs))

    bars = ax.bar(x_pos, config_mean_errors, yerr=config_std_errors, capsize=5,
                alpha=0.75, color=config_colors, edgecolor='black', linewidth=1.5)
    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Position Error (m)', fontsize=11, fontweight='bold')
    ax.set_title('Mean Position Error by config (±1 SD)', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(configs)
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add value labels
    for i, (mean, std) in enumerate(zip(config_mean_errors, config_std_errors)):
        ax.text(i, mean + std + 0.002, f'{mean:.3f}m', ha='center', va='bottom',
                fontsize=9, fontweight='bold')

# 4. Box Plot - Orientation Error by config
ax = axes[1, 0]
if 'config_name' in stats_df.columns:
    config_orient_errors = [stats_df[stats_df['config_name'] == v]['mean_orient_error'].values for v in configs]
    bp = ax.boxplot(config_orient_errors, tick_labels=configs, patch_artist=True, widths=0.6)

    for patch, color in zip(bp['boxes'], config_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.5)

    for element in ['whiskers', 'fliers', 'medians', 'caps']:
        plt.setp(bp[element], color='black', linewidth=1.5)

    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Orientation Error (rad)', fontsize=11, fontweight='bold')
    ax.set_title('Orientation Error Distribution by config', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 5. Scatter: Duration vs Position Error (colored by config)
ax = axes[1, 1]
if 'config_name' in stats_df.columns:
    for i, config in enumerate(configs):
        config_mask = stats_df['config_name'] == config
        durations = []
        errors = []
        for _, row in stats_df[config_mask].iterrows():
            # Get duration for this config-run combination
            dur_mask = (df_duration_summary['config_name'] == row['config_name']) & (df_duration_summary['run_id'] == row['run_id'])
            if dur_mask.any():
                durations.append(df_duration_summary[dur_mask]['duration'].values[0])
                errors.append(row['mean_pos_error'])
        
        if len(durations) > 0:
            ax.scatter(durations, errors, c=[config_colors[i]], s=100, alpha=0.7,
                    edgecolors='black', linewidth=1.5, label=config)

    ax.set_xlabel('Navigation Duration (s)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Position Error (m)', fontsize=11, fontweight='bold')
    ax.set_title('Duration vs Position Error by config', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, ncol=2, loc='best')
    ax.grid(True, alpha=0.3, linewidth=0.5)

# 6. Max Error Comparison by config
ax = axes[1, 2]
if 'config_name' in stats_df.columns:
    config_max_errors = [stats_df[stats_df['config_name'] == v]['max_pos_error'].mean() for v in configs]
    bars = ax.bar(x_pos, config_max_errors, alpha=0.75, color=config_colors,
                edgecolor='black', linewidth=1.5)
    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Max Position Error (m)', fontsize=11, fontweight='bold')
    ax.set_title('Average Maximum Error by config', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(configs)
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add value labels
    for i, max_err in enumerate(config_max_errors):
        ax.text(i, max_err + 0.005, f'{max_err:.3f}m', ha='center', va='bottom',
                fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Speed Analysis

In [ ]:
# Multi-config Speed Analysis
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Box Plot - Linear Speed by config
ax = axes[0, 0]
if 'config_name' in speed_stats_df.columns:
    config_linear_speeds = [speed_stats_df[speed_stats_df['config_name'] == v]['mean_linear'].values for v in configs]
    bp = ax.boxplot(config_linear_speeds, tick_labels=configs, patch_artist=True, widths=0.6)

    for patch, color in zip(bp['boxes'], config_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.5)

    for element in ['whiskers', 'fliers', 'medians', 'caps']:
        plt.setp(bp[element], color='black', linewidth=1.5)

    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Linear Speed (m/s)', fontsize=11, fontweight='bold')
    ax.set_title('Linear Speed Distribution by config', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 2. Heatmap - Mean Linear Speed by config and Run
ax = axes[0, 1]
if 'config_name' in speed_stats_df.columns and 'run_id' in speed_stats_df.columns:
    # NaN, not zero: a (config, run) cell with no measurement is left blank by imshow,
    # where 0.0 would read as a perfect run. And the run ids are used as ids -- stringifying
    # them made every mask below compare str to int, so nothing ever matched and the whole
    # heatmap rendered as zeros, which is a wrong answer rather than a missing one.
    speed_matrix = np.full((len(configs), len(runs)), np.nan)
    sorted_runs_list = sorted(runs)
    for i, config in enumerate(configs):
        for j, run in enumerate(sorted_runs_list):
            mask = (speed_stats_df['config_name'] == config) & (speed_stats_df['run_id'] == run)
            if mask.any():
                speed_matrix[i, j] = speed_stats_df[mask]['mean_linear'].values[0]

    im = ax.imshow(speed_matrix, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(np.arange(len(sorted_runs_list)))
    ax.set_yticks(np.arange(len(configs)))
    ax.set_xticklabels([str(r) for r in sorted_runs_list])
    ax.set_yticklabels(configs)
    ax.set_xlabel('Run Number', fontsize=11, fontweight='bold')
    ax.set_ylabel('config', fontsize=11, fontweight='bold')
    ax.set_title('Mean Linear Speed Heatmap', fontsize=12, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Speed (m/s)', fontsize=10, fontweight='bold')

    # Add text annotations
    for i in range(len(configs)):
        for j in range(len(sorted_runs_list)):
            if speed_matrix[i, j] > 0:
                ax.text(j, i, f'{speed_matrix[i, j]:.2f}',
                    ha="center", va="center", color="black", fontsize=7, fontweight='bold')

# 3. Mean Linear Speed by config with Error Bars
ax = axes[0, 2]
if 'config_name' in speed_stats_df.columns:
    config_mean_speeds = [speed_stats_df[speed_stats_df['config_name'] == v]['mean_linear'].mean() for v in configs]
    config_std_speeds = [speed_stats_df[speed_stats_df['config_name'] == v]['mean_linear'].std() for v in configs]
    x_pos = np.arange(len(configs))

    bars = ax.bar(x_pos, config_mean_speeds, yerr=config_std_speeds, capsize=5,
                alpha=0.75, color=config_colors, edgecolor='black', linewidth=1.5)
    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Linear Speed (m/s)', fontsize=11, fontweight='bold')
    ax.set_title('Mean Linear Speed by config (±1 SD)', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(configs)
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add value labels
    for i, (mean, std) in enumerate(zip(config_mean_speeds, config_std_speeds)):
        if not np.isnan(mean):
            ax.text(i, mean + std + 0.002, f'{mean:.3f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

    # 4. Box Plot - Angular Speed by config
    ax = axes[1, 0]
    config_angular_speeds = [speed_stats_df[speed_stats_df['config_name'] == v]['mean_angular'].values for v in configs]
    bp = ax.boxplot(config_angular_speeds, tick_labels=configs, patch_artist=True, widths=0.6)

    for patch, color in zip(bp['boxes'], config_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.5)

    for element in ['whiskers', 'fliers', 'medians', 'caps']:
        plt.setp(bp[element], color='black', linewidth=1.5)

    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Angular Speed (rad/s)', fontsize=11, fontweight='bold')
    ax.set_title('Angular Speed Distribution by config', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 5. Scatter: Speed vs Error (colored by config)
ax = axes[1, 1]
if 'config_name' in speed_stats_df.columns:
    for i, config in enumerate(configs):
        speeds = []
        errors = []
        for _, row in speed_stats_df[speed_stats_df['config_name'] == config].iterrows():
            # Get error for this config-run combination
            err_mask = (stats_df['config_name'] == row['config_name']) & (stats_df['run_id'] == row['run_id'])
            if err_mask.any():
                speeds.append(row['mean_linear'])
                errors.append(stats_df[err_mask]['mean_pos_error'].values[0])
        
        if len(speeds) > 0:
            ax.scatter(speeds, errors, c=[config_colors[i]], s=100, alpha=0.7,
                    edgecolors='black', linewidth=1.5, label=config)

    ax.set_xlabel('Mean Linear Speed (m/s)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Position Error (m)', fontsize=11, fontweight='bold')
    ax.set_title('Speed vs Position Error by config', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, ncol=2, loc='best')
    ax.grid(True, alpha=0.3, linewidth=0.5)

# 6. Max Speed Comparison by config
ax = axes[1, 2]
if 'config_name' in speed_stats_df.columns:
    config_max_linear = [speed_stats_df[speed_stats_df['config_name'] == v]['max_linear'].mean() for v in configs]
    config_max_angular = [speed_stats_df[speed_stats_df['config_name'] == v]['max_angular'].mean() for v in configs]

    width = 0.35
    x = np.arange(len(configs))
    ax.bar(x - width/2, config_max_linear, width, label='Linear', alpha=0.75,
        color=[c for c in config_colors], edgecolor='black', linewidth=1.5)
    ax.bar(x + width/2, config_max_angular, width, label='Angular', alpha=0.75,
        color=[c for c in config_colors], edgecolor='black', linewidth=1.5,
        hatch='//')

    ax.set_xlabel('config', fontsize=11, fontweight='bold')
    ax.set_ylabel('Max Speed', fontsize=11, fontweight='bold')
    ax.set_title('Average Maximum Speeds by config', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.5)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add secondary y-axis labels
    ax2 = ax.twinx()
    ax2.set_ylabel('(Linear: m/s, Angular: rad/s)', fontsize=10, style='italic')
    ax2.set_yticks([])

plt.tight_layout()
plt.show()